### RaschPy conditional Rasch analysis: worked example (PCM)

A conditional analysis in the style of Christensen, Kreiner & Mesbah (2013): every step conditions on each person's total score, so the person parameters drop out. The steps are CML estimation, overall fit, item fit, local dependence and dimensionality, DIF, person fit and targeting, then a choice between the partial credit and rating scale models.

#### Data

This example uses the DESC-II depression screening data from **iarm**, an R package for Rasch analysis (Mueller, 2022; https://CRAN.R-project.org/package=iarm). The data are 799 patients' responses to 10 items scored from 0 (never) to 4 (always), with each patient's indication group, gender and age group. To run the notebook, export the data once from R into the working directory:

```r
install.packages("iarm")
write.csv(iarm::desc2, "desc2.csv", row.names = FALSE)
```

In [1]:
import os
import numpy as np
import pandas as pd
from raschpy import PCM, RSM

os.chdir('my_working_directory')

desc2 = pd.read_csv('desc2.csv')
items = [c for c in desc2.columns if c.startswith('DESC')]
responses = desc2[items]
exogenous = desc2[['group', 'gender', 'agegroup']]

#### 1. CML estimation

`estimation='cml'` gives conditional maximum likelihood estimates. For the PCM, CML has a ridge by default; `ridge=0` gives plain CML, whose estimates, infit and outfit, item-restscore and Martin-Löf results agree with iarm's to the decimals it prints.

In [2]:
pcm = PCM(responses,
          max_score_vector=[4] * len(items),
          exogenous=exogenous)
pcm.calibrate(estimation='cml')
pcm.std_errors()  # sandwich standard errors

round(pd.DataFrame({'Location': pcm.items, 'SE': pcm.item_se}), 3)

,Location,SE
DESC_2_1,0.117,0.060
DESC_2_2,0.452,0.070
DESC_2_3,-0.893,0.054
DESC_2_4,-0.565,0.055
DESC_2_5,0.347,0.056
DESC_2_6,0.145,0.061
DESC_2_7,-0.057,0.053
DESC_2_8,-0.223,0.052
DESC_2_9,-0.555,0.057
DESC_2_10,1.231,0.111


In [3]:
round(pcm.thresholds_uncentred, 3)

,1,2,3,4
DESC_2_1,-0.950,-0.778,0.661,1.533
DESC_2_2,-0.594,-0.538,0.974,1.968
DESC_2_3,-3.400,-1.649,0.091,1.387
DESC_2_4,-2.611,-1.074,0.070,1.354
DESC_2_5,-0.320,-0.392,0.396,1.704
DESC_2_6,-1.607,-0.435,0.485,2.137
DESC_2_7,-1.180,-0.825,0.419,1.358
DESC_2_8,-2.116,-1.009,0.368,1.864
DESC_2_9,-2.385,-1.439,-0.085,1.690
DESC_2_10,0.754,0.392,1.652,2.127


Standard errors from the inverse Hessian instead, as most CML software reports them:

In [4]:
pcm.std_errors(se_method='hessian')
hessian_se = pcm.item_se
pcm.std_errors()

round(pd.DataFrame({'Sandwich': pcm.item_se, 'Hessian': hessian_se}), 3)

,Sandwich,Hessian
DESC_2_1,0.060,0.058
DESC_2_2,0.070,0.066
DESC_2_3,0.054,0.058
DESC_2_4,0.055,0.056
DESC_2_5,0.056,0.060
DESC_2_6,0.061,0.064
DESC_2_7,0.053,0.056
DESC_2_8,0.052,0.060
DESC_2_9,0.057,0.057
DESC_2_10,0.111,0.086


#### 2. Overall fit

Andersen's conditional likelihood ratio test, splitting the persons by total score and by gender. With the PCM ridge, the Monte Carlo *p*-value is the one to read.

In [5]:
pcm.global_fit(method='andersen',
               split='score',
               seed=1)
pcm.global_fit_summary

Andersen LR Monte Carlo p-value: 1/500 samples | elapsed 0s | about 59s left

Andersen LR Monte Carlo p-value: 6/500 samples | elapsed 1s | about 58s left

Andersen LR Monte Carlo p-value: 11/500 samples | elapsed 1s | about 57s left

Andersen LR Monte Carlo p-value: 16/500 samples | elapsed 2s | about 57s left

Andersen LR Monte Carlo p-value: 21/500 samples | elapsed 2s | about 56s left

Andersen LR Monte Carlo p-value: 26/500 samples | elapsed 3s | about 54s left

... (26 outputs left out)

Andersen LR Monte Carlo p-value: 158/500 samples | elapsed 18s | about 40s left

Andersen LR Monte Carlo p-value: 163/500 samples | elapsed 19s | about 39s left

Andersen LR Monte Carlo p-value: 165/500 samples | elapsed 19s | about 39s left | done

method                                                                  andersen
split                                                                      score
statistic                                                              35.672963
df                                                                            39
split_prop                                                                   0.5
log-likelihoods                {'low': -1119.0075243231945, 'high': -3716.077...
p_method                                                             monte-carlo
p-value                                                                 0.060241
p-value (chi-square)                                                    0.622429
Monte Carlo samples                                                          166
Monte Carlo samples skipped                                                    0
Monte Carlo stopped early                                                   True
reject                      

In [6]:
pcm.global_fit(method='andersen',
               split='exogenous',
               covariate='gender',
               seed=1)
pcm.global_fit_summary

Andersen LR Monte Carlo p-value: 1/500 samples | elapsed 0s | about 19s left

Andersen LR Monte Carlo p-value: 14/500 samples | elapsed 1s | about 19s left

Andersen LR Monte Carlo p-value: 28/500 samples | elapsed 1s | about 18s left

Andersen LR Monte Carlo p-value: 41/500 samples | elapsed 2s | about 18s left

Andersen LR Monte Carlo p-value: 54/500 samples | elapsed 2s | about 17s left

Andersen LR Monte Carlo p-value: 67/500 samples | elapsed 3s | about 17s left

... (32 outputs left out)

Andersen LR Monte Carlo p-value: 499/500 samples | elapsed 20s | about 0s left

Andersen LR Monte Carlo p-value: 500/500 samples | elapsed 20s | about 0s left

Andersen LR Monte Carlo p-value: 500/500 samples | elapsed 20s | about 0s left | done

method                                                                  andersen
split                                                                  exogenous
statistic                                                              79.119529
df                                                                            39
covariate                                                                 gender
log-likelihoods                {'female': -2436.0507045146333, 'male': -2371....
p_method                                                             monte-carlo
p-value                                                                 0.001996
p-value (chi-square)                                                    0.000153
Monte Carlo samples                                                          500
Monte Carlo samples skipped                                                    0
Monte Carlo stopped early                                                  False
reject                      

Martin-Löf's test of unidimensionality, splitting the items at their median location. Most cells of its score table are sparse here, so the Monte Carlo *p*-value is used:

In [7]:
pcm.global_fit(method='martin-lof',
               p_method='monte-carlo',
               seed=1)
pcm.global_fit_summary

Martin-Lof Monte Carlo p-value: 1/500 samples | elapsed 0s | about 17s left

Martin-Lof Monte Carlo p-value: 16/500 samples | elapsed 1s | about 17s left

Martin-Lof Monte Carlo p-value: 31/500 samples | elapsed 1s | about 16s left

Martin-Lof Monte Carlo p-value: 47/500 samples | elapsed 2s | about 15s left

Martin-Lof Monte Carlo p-value: 62/500 samples | elapsed 2s | about 15s left

Martin-Lof Monte Carlo p-value: 77/500 samples | elapsed 3s | about 14s left

... (26 outputs left out)

Martin-Lof Monte Carlo p-value: 488/500 samples | elapsed 17s | about 0s left

Martin-Lof Monte Carlo p-value: 500/500 samples | elapsed 17s | about 0s left

Martin-Lof Monte Carlo p-value: 500/500 samples | elapsed 17s | about 0s left | done

method                                                                 martin-lof
split                                                                      median
statistic                                                              351.430344
df                                                                            399
subsets                         {'A': ['DESC_2_3', 'DESC_2_4', 'DESC_2_7', 'DE...
log-likelihoods                 {'A': -2104.5257490051645, 'B': -1711.01020892...
p_method                                                              monte-carlo
p-value                                                                  0.001996
p-value (chi-square)                                                     0.958372
Monte Carlo samples                                                           500
Monte Carlo samples skipped                                                     0
Monte Carlo stopped early                                                   False
reject          

#### 3. Item fit

Conditional infit and outfit, item-restscore and the conditional item chi-square Q1, with *p*-values and cutoffs from 200 datasets simulated from the model given each person's total, and Westfall–Young familywise *p*-values:

In [8]:
pcm.fit_statistics(conditional=True,
                   bootstrap_samples=200,
                   seed=1)
pcm.item_stats_df(full=True)

,Estimate,SE,2.5%,97.5%,Count,Facility,Infit MS,Infit Z,Outfit MS,Outfit Z,PM corr,Exp PM corr,Observed gamma,Expected gamma,Restscore p (BH),Q1,Q1 df,Q1 p (BH),Infit p (Westfall-Young),Outfit p (Westfall-Young)
DESC_2_1,0.117,0.060,-0.001,0.235,799,0.230,1.097,1.543,1.231,2.325,0.772,0.737,0.776,0.796,0.196,5.960,9,0.827,0.398,0.090
DESC_2_2,0.452,0.070,0.316,0.589,799,0.195,1.101,1.543,1.152,1.340,0.752,0.718,0.790,0.798,0.647,10.089,9,0.731,0.398,0.687
DESC_2_3,-0.893,0.054,-0.999,-0.786,799,0.368,0.889,-2.166,0.906,-1.922,0.825,0.725,0.801,0.779,0.196,8.604,9,0.731,0.219,0.328
DESC_2_4,-0.565,0.055,-0.674,-0.457,799,0.322,1.063,1.155,1.067,1.246,0.802,0.743,0.773,0.782,0.467,5.957,9,0.827,0.493,0.687
DESC_2_5,0.347,0.056,0.238,0.456,799,0.197,0.899,-1.515,0.910,-0.681,0.778,0.720,0.827,0.807,0.264,8.225,9,0.731,0.458,0.791
DESC_2_6,0.145,0.061,0.026,0.265,799,0.241,0.978,-0.365,1.011,0.184,0.788,0.731,0.779,0.783,0.728,4.255,9,0.894,0.667,0.935
DESC_2_7,-0.057,0.053,-0.160,0.047,799,0.250,0.910,-1.507,0.848,-1.864,0.816,0.744,0.823,0.794,0.134,14.137,9,0.471,0.493,0.328
DESC_2_8,-0.223,0.052,-0.326,-0.120,799,0.287,0.797,-3.917,0.796,-3.717,0.843,0.740,0.834,0.782,0.000,13.502,9,0.471,0.005,0.015
DESC_2_9,-0.555,0.057,-0.667,-0.443,799,0.329,1.070,1.273,1.085,1.517,0.805,0.746,0.769,0.784,0.264,9.310,9,0.731,0.493,0.393
DESC_2_10,1.231,0.111,1.014,1.448,799,0.104,1.511,5.342,1.082,0.442,0.602,0.620,0.771,0.806,0.171,37.322,9,0.000,0.005,0.935


In [9]:
round(pcm.item_restscore_table, 3)

,Observed gamma,Expected gamma,SE,Simulated mean,Simulated SD,z,p,p (BH),p (bootstrap),"p (bootstrap, Westfall-Young)"
DESC_2_1,0.776,0.796,0.017,0.797,0.013,-1.654,0.098,0.196,0.060,0.328
DESC_2_2,0.790,0.798,0.018,0.798,0.014,-0.550,0.583,0.647,0.552,0.791
DESC_2_3,0.801,0.779,0.014,0.781,0.011,1.711,0.087,0.196,0.129,0.443
DESC_2_4,0.773,0.782,0.016,0.783,0.012,-0.889,0.374,0.467,0.433,0.791
DESC_2_5,0.827,0.807,0.015,0.809,0.014,1.327,0.185,0.264,0.129,0.468
DESC_2_6,0.779,0.783,0.016,0.784,0.013,-0.348,0.728,0.728,0.627,0.791
DESC_2_7,0.823,0.794,0.015,0.796,0.012,2.214,0.027,0.134,0.050,0.224
DESC_2_8,0.834,0.782,0.013,0.783,0.012,4.374,0.000,0.000,0.005,0.005
DESC_2_9,0.769,0.784,0.015,0.786,0.012,-1.395,0.163,0.264,0.144,0.488
DESC_2_10,0.771,0.806,0.021,0.808,0.019,-1.948,0.051,0.171,0.035,0.264


In [10]:
round(pcm.item_q1_table, 3)

,Q1,df,p,p (BH),p (bootstrap),"p (bootstrap, Westfall-Young)"
DESC_2_1,5.960,9,0.744,0.827,0.706,0.945
DESC_2_2,10.089,9,0.343,0.731,0.264,0.886
DESC_2_3,8.604,9,0.475,0.731,0.348,0.886
DESC_2_4,5.957,9,0.744,0.827,0.672,0.945
DESC_2_5,8.225,9,0.512,0.731,0.403,0.886
DESC_2_6,4.255,9,0.894,0.894,0.831,0.945
DESC_2_7,14.137,9,0.118,0.471,0.095,0.507
DESC_2_8,13.502,9,0.141,0.471,0.090,0.483
DESC_2_9,9.310,9,0.409,0.731,0.313,0.886
DESC_2_10,37.322,9,0.000,0.000,0.010,0.015


In [11]:
round(pcm.item_fit_bootstrap, 3)

,Infit MS,Infit 95% low,Infit 95% high,Infit p,Infit p (Westfall-Young),Outfit MS,Outfit 95% low,Outfit 95% high,Outfit p,Outfit p (Westfall-Young)
DESC_2_1,1.097,0.892,1.086,0.070,0.398,1.231,0.836,1.138,0.010,0.090
DESC_2_2,1.101,0.891,1.099,0.065,0.398,1.152,0.777,1.212,0.199,0.687
DESC_2_3,0.889,0.913,1.109,0.040,0.219,0.906,0.908,1.091,0.060,0.328
DESC_2_4,1.063,0.903,1.094,0.259,0.493,1.067,0.900,1.097,0.229,0.687
DESC_2_5,0.899,0.884,1.127,0.119,0.458,0.910,0.807,1.227,0.433,0.791
DESC_2_6,0.978,0.887,1.081,0.667,0.667,1.011,0.894,1.111,0.736,0.935
DESC_2_7,0.910,0.886,1.105,0.169,0.493,0.848,0.865,1.156,0.060,0.328
DESC_2_8,0.797,0.910,1.083,0.005,0.005,0.796,0.902,1.114,0.005,0.015
DESC_2_9,1.070,0.899,1.088,0.164,0.493,1.085,0.912,1.091,0.065,0.393
DESC_2_10,1.511,0.863,1.135,0.005,0.005,1.082,0.644,1.356,0.796,0.935


#### 4. Local dependence and dimensionality

Q3\* (residual correlations minus their mean), with the simulated cutoff, and the first contrast of the residuals' principal components:

In [12]:
pcm.res_corr_analysis()
round(pcm.q3_star, 3)

,DESC_2_1,DESC_2_2,DESC_2_3,DESC_2_4,DESC_2_5,DESC_2_6,DESC_2_7,DESC_2_8,DESC_2_9,DESC_2_10
DESC_2_1,1.111,0.022,-0.073,-0.058,0.191,-0.015,-0.031,-0.134,-0.074,0.095
DESC_2_2,0.022,1.111,-0.054,0.001,-0.084,0.091,-0.063,-0.060,0.096,0.001
DESC_2_3,-0.073,-0.054,1.111,0.049,-0.079,-0.085,0.100,0.225,-0.020,0.021
DESC_2_4,-0.058,0.001,0.049,1.111,-0.102,0.051,0.036,0.050,0.032,-0.123
DESC_2_5,0.191,-0.084,-0.079,-0.102,1.111,0.062,-0.050,0.042,-0.028,0.095
DESC_2_6,-0.015,0.091,-0.085,0.051,0.062,1.111,0.001,-0.014,0.004,-0.086
DESC_2_7,-0.031,-0.063,0.100,0.036,-0.050,0.001,1.111,0.084,-0.079,0.031
DESC_2_8,-0.134,-0.060,0.225,0.050,0.042,-0.014,0.084,1.111,-0.003,-0.062
DESC_2_9,-0.074,0.096,-0.020,0.032,-0.028,0.004,-0.079,-0.003,1.111,-0.005
DESC_2_10,0.095,0.001,0.021,-0.123,0.095,-0.086,0.031,-0.062,-0.005,1.111


In [13]:
round(pcm.q3_bootstrap, 3)

,Item 1,Item 2,Q3,Expected Q3,95% cutoff,p,p (Westfall-Young)
0,DESC_2_3,DESC_2_8,0.115,-0.122,-0.053,0.005,0.005
1,DESC_2_1,DESC_2_5,0.080,-0.124,-0.040,0.005,0.005
2,DESC_2_3,DESC_2_7,-0.011,-0.115,-0.046,0.005,0.234
3,DESC_2_2,DESC_2_9,-0.015,-0.104,-0.036,0.030,0.512
4,DESC_2_1,DESC_2_10,-0.016,-0.106,-0.032,0.020,0.572
5,DESC_2_5,DESC_2_10,-0.016,-0.113,-0.035,0.030,0.706
6,DESC_2_2,DESC_2_6,-0.019,-0.111,-0.045,0.025,0.622
7,DESC_2_7,DESC_2_8,-0.026,-0.115,-0.050,0.015,0.478
8,DESC_2_5,DESC_2_6,-0.048,-0.117,-0.034,0.080,0.935
9,DESC_2_4,DESC_2_6,-0.060,-0.112,-0.047,0.104,0.985


In [14]:
pcm.residual_contrast_bootstrap

{'first_contrast': 1.5384832757312312,
 'expected': 1.290614582080172,
 'cutoff_95': 1.3458322106810958,
 'p': 0.004975124378109453}

Partial gamma tests of local dependence (Kreiner & Christensen, 2004):

In [15]:
round(pcm.local_dependence_gamma(), 3)

,Item,Paired item,Gamma,z,p,p (BH),Flagged
0,DESC_2_3,DESC_2_8,0.491,7.009,0.000,0.000,True
1,DESC_2_8,DESC_2_3,0.445,6.160,0.000,0.000,True
2,DESC_2_1,DESC_2_5,0.308,4.365,0.000,0.000,True
3,DESC_2_5,DESC_2_1,0.295,4.068,0.000,0.001,True
4,DESC_2_4,DESC_2_7,0.217,3.093,0.002,0.020,True
...,...,...,...,...,...,...,...
85,DESC_2_3,DESC_2_1,-0.242,-3.168,0.002,0.017,True
86,DESC_2_10,DESC_2_4,-0.245,-2.687,0.007,0.050,True
87,DESC_2_4,DESC_2_10,-0.283,-3.242,0.001,0.015,True
88,DESC_2_4,DESC_2_5,-0.287,-3.571,0.000,0.005,True


#### 5. Differential item functioning

DIF by gender, with the conditional likelihood ratio omnibus test:

In [16]:
pcm.dif_test('gender',
             seed=1)
round(pcm.dif_table, 3)

/Users/markelliott/Documents/RaschPy-dev/raschpy/_progress.py:67: UserWarning: 1 non-extreme person(s) have a missing value for 'gender' and were excluded. Group sizes used: {'male': 348, 'female': 322}.
  return func(*args, **kwargs)


,Group,Reference,Focal,Focal (purified),Difference,SE,t,p,p (corrected),Selected,Flagged,df
Item,,,,,,,,,,,,
DESC_2_1,female,0.115,0.113,0.245,0.130,0.145,0.902,0.367,0.476,True,False,663.524
DESC_2_2,female,0.277,0.591,0.709,0.432,0.163,2.651,0.008,0.067,False,False,667.990
DESC_2_3,female,-0.773,-1.060,-0.969,-0.197,0.130,-1.508,0.132,0.330,True,False,665.724
DESC_2_4,female,-0.551,-0.581,-0.450,0.100,0.125,0.805,0.421,0.476,True,False,666.068
DESC_2_5,female,0.342,0.351,0.486,0.145,0.139,1.040,0.299,0.476,True,False,647.446
DESC_2_6,female,0.120,0.157,0.294,0.173,0.142,1.224,0.221,0.443,True,False,656.410
DESC_2_7,female,0.043,-0.156,-0.053,-0.096,0.121,-0.792,0.429,0.476,True,False,667.388
DESC_2_8,female,-0.056,-0.350,-0.259,-0.203,0.130,-1.563,0.119,0.330,True,False,588.218
DESC_2_9,female,-0.496,-0.600,-0.480,0.016,0.139,0.112,0.911,0.911,True,False,649.591


In [17]:
pcm.dif_omnibus_table

,LR,df,p,Flagged
female,46.723701,9,0.0,True


#### 6. Person fit

Each person's conditional *p*-value: the probability, given their total score, of a response pattern as unlikely as theirs:

In [18]:
p = pcm.person_conditional_p
print(f"{(p < 0.05).sum()} of {p.notna().sum()} persons have p < 0.05")
round(p.sort_values().head(10), 4)

59 of 671 persons have p < 0.05


512    0.001
272    0.001
495    0.001
527    0.001
82     0.001
171    0.001
60     0.001
129    0.001
58     0.002
327    0.002
Name: Conditional p, dtype: float64

#### 7. Targeting and reliability

In [19]:
pcm.targeting()
pcm.targeting_summary

Population mean                  -1.997563
Population SD                     2.189210
Target                            0.008356
Target - population mean          2.005919
Maximum information               8.429535
Expected information              4.233059
Expected / maximum information    0.502170
Test difficulty                   0.029823
Expected |bias|                   0.129513
Expected SEM                      0.528693
Expected RMSE                     0.605380
rho1                              0.959796
rho2                              0.959796
rho3                              0.926769
rho4                              0.926769
rho5                              0.950038
rho6                              0.889509
Name: Targeting, dtype: float64

#### 8. Summary

The results above in one table:

In [20]:
pcm.global_stats_df()

,Section,Value,df,p,Details
Statistic,,,,,
Overall Q1 (item scores),Overall fit,105.626,NaN,0.010,"10 ability groups; Monte Carlo p, 99 samples"
Overall Q1 (thresholds),Overall fit,676.619,NaN,0.020,"10 ability groups; Monte Carlo p, 99 samples"
Andersen LR,Overall fit,35.673,39.0,0.060,split=score; p_method=monte-carlo
Andersen LR,Overall fit,79.120,39.0,0.002,split=exogenous; covariate=gender; p_method=mo...
Martin-Lof,Dimensionality and local dependence,351.430,399.0,0.002,split=median; p_method=monte-carlo; 91% of sco...
First residual contrast,Dimensionality and local dependence,1.538,NaN,0.005,bootstrap 95% cutoff 1.346
Q3* (largest),Dimensionality and local dependence,0.225,NaN,NaN,conditional residuals
Item pairs with local dependence,Dimensionality and local dependence,6.000,NaN,NaN,"of 90 ordered pairs; partial gamma, BH p <= 0...."
Items flagged by Q1,Item fit,1.000,NaN,NaN,of 10; BH p <= 0.05


#### 9. Partial credit or rating scale model?

The rating scale model is nested in the partial credit model; with CML estimates, `model_selection(test='LR')` tests it by a conditional likelihood ratio test (the default, `test=None`, uses AIC). With many persons any difference between the two is significant, so the partial credit model is preferred only if a close-fit test also rejects the rating scale model: its null hypothesis is that the partial credit model's thresholds depart from the rating scale model's by at most `delta` (default 0.1 median category widths), tested on the root mean square departure over all the items' thresholds (`p-value (close fit)`) and on the item that departs most (`per item`). `Delta-hat` estimates the departure, in median category widths:

In [21]:
rsm = RSM(responses,
          max_score=4,
          exogenous=exogenous)
rsm.calibrate(estimation='cml')
rsm.model_selection(test='LR')
rsm.model_comparison_rsm_pcm_lr_summary

LR statistic                     286.473231
df                                       27
p-value                                 0.0
delta                                   0.1
p-value (close fit)                     0.0
p-value (close fit, per item)           0.0
Delta-hat                           0.28174
Units                                 width
Preferred                               PCM
Name: RSM vs PCM LR test, dtype: object

#### References

Christensen, K. B., Kreiner, S. & Mesbah, M. (Eds.) (2013). *Rasch Models in Health*. Wiley.

Kreiner, S. & Christensen, K. B. (2004). Analysis of local dependence and multidimensionality in graphical loglinear Rasch models. *Communications in Statistics – Theory and Methods*, 33, 1239–1276.

Mueller, M. (2022). *iarm: Item Analysis in Rasch Models*. R package version 0.4.3. https://CRAN.R-project.org/package=iarm